In [14]:
import os
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [15]:
from sparkSession import spark
from pyspark.sql.functions import *
from silver_schema_def import schema_silver_events

In [16]:
dataframe = spark.read.json('/Users/ayusman/thisWorks/DE/lakehouse_analytics/data/streaming')

In [17]:
READ_PATH_CLICKSTREAM = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_clickstream'
READ_PATH_USER = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_user'

In [18]:
dataframe_clickstream = spark.read.format('delta').load(READ_PATH_CLICKSTREAM)
dataframe_user = spark.read.format('delta').load(READ_PATH_USER)

In [19]:
dataframe_clickstream.printSchema()
dataframe_user.printSchema()

root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)

root
 |-- change_type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- email: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [20]:
## checking for nulls across each column
'''print("Null counts across columns")
for colss in dataframe.columns:
    df_null = dataframe.filter(col(colss).isNull()).count()
    print(f"{colss}: {df_null}") '''

'print("Null counts across columns")\nfor colss in dataframe.columns:\n    df_null = dataframe.filter(col(colss).isNull()).count()\n    print(f"{colss}: {df_null}") '

In [21]:
dataframe_clickstream.printSchema()

root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [22]:
dataframe_clickstream.show(2)

+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+--------------------+--------------------+
|    category|country| device|            event_id|          event_time|event_type| price|product_id|quantity| session_id|user_id|        ingestion_ts|        ip_file_name|
+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+--------------------+--------------------+
|prescription|Germany|desktop|3ff90190-f6e2-48c...|2026-09-19T21:03:...|  checkout|137.73|   prod_14|       4|session_407|user_75|2026-10-03 19:30:...|file:///Users/ayu...|
|    cold_flu| France| mobile|1b8ba84c-525c-46f...|2026-09-09T17:15:...| page_view| 96.08|   prod_19|    NULL|session_430|user_97|2026-10-03 19:30:...|file:///Users/ayu...|
+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+-----

In [23]:
## trim everything everywhere
df_cs_trimmed = dataframe_clickstream.withColumns({c: trim(col(c)) for c in dataframe_clickstream.columns})
df_cs_trimmed.show(2)
df_cs_trimmed.printSchema()

+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+--------------------+--------------------+
|    category|country| device|            event_id|          event_time|event_type| price|product_id|quantity| session_id|user_id|        ingestion_ts|        ip_file_name|
+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+--------------------+--------------------+
|prescription|Germany|desktop|3ff90190-f6e2-48c...|2026-09-19T21:03:...|  checkout|137.73|   prod_14|       4|session_407|user_75|2026-10-03 19:30:...|file:///Users/ayu...|
|    cold_flu| France| mobile|1b8ba84c-525c-46f...|2026-09-09T17:15:...| page_view| 96.08|   prod_19|    NULL|session_430|user_97|2026-10-03 19:30:...|file:///Users/ayu...|
+------------+-------+-------+--------------------+--------------------+----------+------+----------+--------+-----------+-------+-----

In [24]:
## changing existing bronze layer schema to type enforced schema 
df_cs = spark.createDataFrame(dataframe_clickstream.rdd, schema=schema_silver_events)
df_cs.printSchema()

root
 |-- event_id: string (nullable = false)
 |-- session_id: string (nullable = false)
 |-- user_id: string (nullable = true)
 |-- user_sk: long (nullable = false)
 |-- product_id: string (nullable = true)
 |-- product_sk: long (nullable = true)
 |-- event_ts: timestamp (nullable = false)
 |-- event_date: date (nullable = false)
 |-- event_type: string (nullable = false)
 |-- is_logged_in: boolean (nullable = false)
 |-- is_bot: boolean (nullable = false)
 |-- category: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- search_query: string (nullable = true)
 |-- device: string (nullable = false)
 |-- country: string (nullable = false)
 |-- traffic_source: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- ip_hash: string (nullable = true)
 |-- ingest_ts: timestamp (nullabl

In [25]:
df_cs = df_cs.dropna(how='any', subset=['session_id', 'event_ts', 'event_type', 'product_id', 'event_id'])
df_cs.count()

26/10/03 21:15:40 ERROR Executor: Exception in task 0.0 in stage 25.0 (TID 359)
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3622, in main
    process()
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/util.py", line 159, in wrapper
    return f(*args, **kwargs)
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/sql/session.py", line 1675, in prepare
    verify_func(obj)
  File "/opt/home

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3622, in main
    process()
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/util.py", line 159, in wrapper
    return f(*args, **kwargs)
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/sql/session.py", line 1675, in prepare
    verify_func(obj)
  File "/opt/homebrew/lib/python3.10/site-packages/pyspark/sql/types.py", line 3470, in verify
    verify_value(obj)
  File "/opt/homebrew/lib/p